In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HOQ_H.xpt"
csv_file = DATA_DIR / "HOQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched source for fidelity testing.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 HOQ_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HOD050",
    "HOQ065"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HOQ_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 10175:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,175 rows x 3 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# HOQ_H contains no genuine continuous decimal variables.
#
# SEQN:
#     Respondent identifier
#
# HOD050:
#     Number of rooms in home / special codes
#
# HOQ065:
#     Housing arrangement response code
#
# Therefore all three can safely become nullable Int64.
#
# IMPORTANT:
#
# This applies only to HOQ_H.
#
# Genuine Dietary / Laboratory / Examination decimals
# in other NHANES files must remain exactly as released.
#

DECIMAL_COLUMNS = []


INTEGER_COLUMNS = [
    "SEQN",
    "HOD050",
    "HOQ065"
]


if len(INTEGER_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly 3 whole-number/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code variables:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached HOQ_H contains NONE.
#
# Therefore no tiny -> zero restoration is necessary.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 3 HOQ_H variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Tiny XPORT artifacts:",
    tiny_total
)


print(
    "Ordinary numeric zeros:",
    true_zero_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HOQ_H."
    )


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary zero values "
        "found in HOQ_H."
    )


print(
    "PASS: HOQ_H contains no tiny artifacts "
    "and no ordinary zeros."
)


# ============================================================
# 6. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() here is ONLY used as a validation test.
#
# It does NOT modify the data.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in HOQ_H. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 8. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# Convert only after proving no fractional values exist.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 3 HOQ_H variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 9. CDC CHECK - HOD050
# ============================================================

#
# Number of rooms in home.
#
# CDC:
#
# 1      = 88
# 2      = 202
# 3      = 683
# 4      = 1,613
# 5      = 2,093
# 6      = 1,922
# 7      = 1,272
# 8      = 853
# 9      = 574
# 10     = 343
# 11     = 197
# 12     = 108
# 13     = 85       # 13 or more
# 777    = 16       # Refused
# 999    = 5        # Don't know
# Missing = 121
#
# CDC grouped all responses >=13 into code 13.
#

EXPECTED_HOD050 = {
    1: 88,
    2: 202,
    3: 683,
    4: 1613,
    5: 2093,
    6: 1922,
    7: 1272,
    8: 853,
    9: 574,
    10: 343,
    11: 197,
    12: 108,
    13: 85,
    777: 16,
    999: 5
}


actual_hod050 = (
    df["HOD050"]
    .value_counts()
    .sort_index()
    .to_dict()
)


hod050_missing = int(
    df["HOD050"]
    .isna()
    .sum()
)


print("\n--- HOD050 ---")


print(
    "Observed:",
    actual_hod050
)


print(
    "Missing:",
    hod050_missing
)


if actual_hod050 != EXPECTED_HOD050:

    raise ValueError(
        "HOD050 frequencies differ from CDC.\n"
        f"Expected: {EXPECTED_HOD050}\n"
        f"Actual:   {actual_hod050}"
    )


if hod050_missing != 121:

    raise ValueError(
        "Unexpected HOD050 missing count."
    )


print(
    "PASS: HOD050 matches CDC exactly."
)


# ============================================================
# 10. HOD050 ALLOWED-CODE CHECK
# ============================================================

ALLOWED_HOD050 = {
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    777,
    999
}


observed_hod050 = set(
    df["HOD050"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


unexpected_hod050 = (
    observed_hod050
    -
    ALLOWED_HOD050
)


if unexpected_hod050:

    raise ValueError(
        "HOD050 contains unexpected codes: "
        + str(
            sorted(
                unexpected_hod050
            )
        )
    )


print(
    "PASS: HOD050 contains only "
    "CDC-documented values."
)


# ============================================================
# 11. CDC CHECK - HOQ065
# ============================================================

#
# Home owned, bought, rented, other.
#
# CDC:
#
# 1 = Owned or being bought = 5,550
# 2 = Rented                = 4,269
# 3 = Other arrangement      = 223
# 7 = Refused                 = 11
# 9 = Don't know               = 1
#
# Missing = 121
#

EXPECTED_HOQ065 = {
    1: 5550,
    2: 4269,
    3: 223,
    7: 11,
    9: 1
}


actual_hoq065 = (
    df["HOQ065"]
    .value_counts()
    .sort_index()
    .to_dict()
)


hoq065_missing = int(
    df["HOQ065"]
    .isna()
    .sum()
)


print("\n--- HOQ065 ---")


print(
    "Observed:",
    actual_hoq065
)


print(
    "Missing:",
    hoq065_missing
)


if actual_hoq065 != EXPECTED_HOQ065:

    raise ValueError(
        "HOQ065 frequencies differ from CDC.\n"
        f"Expected: {EXPECTED_HOQ065}\n"
        f"Actual:   {actual_hoq065}"
    )


if hoq065_missing != 121:

    raise ValueError(
        "Unexpected HOQ065 missing count."
    )


print(
    "PASS: HOQ065 matches CDC exactly."
)


# ============================================================
# 12. HOQ065 ALLOWED-CODE CHECK
# ============================================================

ALLOWED_HOQ065 = {
    1,
    2,
    3,
    7,
    9
}


observed_hoq065 = set(
    df["HOQ065"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


unexpected_hoq065 = (
    observed_hoq065
    -
    ALLOWED_HOQ065
)


if unexpected_hoq065:

    raise ValueError(
        "HOQ065 contains unexpected codes: "
        + str(
            sorted(
                unexpected_hoq065
            )
        )
    )


print(
    "PASS: HOQ065 contains only "
    "CDC-documented response codes."
)


# ============================================================
# 13. MISSING-PATTERN CONSISTENCY
# ============================================================

#
# The attached release has exactly 121 missing values
# in both household variables.
#
# Verify that those missing positions are identical.
#

hod050_missing_mask = (
    df["HOD050"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


hoq065_missing_mask = (
    df["HOQ065"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


missing_pattern_mismatch = int(
    np.sum(
        hod050_missing_mask
        !=
        hoq065_missing_mask
    )
)


print(
    "\n--- HOQ MISSING-PATTERN CHECK ---"
)


print(
    "HOD050/HOQ065 missing-position mismatches:",
    missing_pattern_mismatch
)


if missing_pattern_mismatch != 0:

    raise ValueError(
        "HOD050 and HOQ065 missing positions differ."
    )


print(
    "PASS: HOD050 and HOQ065 missing "
    "positions match exactly."
)


# ============================================================
# 14. CONFIRM NO ZERO VALUES WERE CREATED
# ============================================================

#
# Original XPT:
#
# tiny values   = 0
# ordinary zero = 0
#
# Therefore there should still be no zero values.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO CHECK ---")


print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values were created."
)


# ============================================================
# 15. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HOQ_H requires NO numeric correction.
#
# Therefore every numeric value must remain
# identical to the original XPT.
#
# Changing:
#
#     73557.0 -> 73557
#     5.0     -> 5
#
# is presentation only.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 3 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 16. EXPORT CSV
# ============================================================

#
# Standard NHANES fidelity export.
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
# HOQ_H itself contains no genuine decimals,
# but keeping this export convention avoids accidental
# damage when the same workflow is used for mixed datasets.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 17. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 18. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 3 CSV variables round-trip "
    "to exact source XPT values."
)


# ============================================================
# 19. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 20. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 21. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 22. RAW CSV .0 CHECK
# ============================================================

#
# HOQ_H contains NO genuine decimal variables.
#
# Therefore it is safe to check all 3 variables
# for unnecessary ".0".
#
# Examples:
#
# SEQN
#     73557.0 -> 73557
#
# HOD050
#     5.0 -> 5
#
# HOQ065
#     1.0 -> 1
#
# DO NOT use this global rule with mixed decimal datasets.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any HOQ_H field."
)


# ============================================================
# 23. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO CHECK ---")


print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found "
        "in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 24. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 25. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "HOQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - Housing Characteristics"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "HOD050 nonmissing:",
    int(
        df["HOD050"]
        .notna()
        .sum()
    )
)


print(
    "HOQ065 nonmissing:",
    int(
        df["HOQ065"]
        .notna()
        .sum()
    )
)


print(
    "Homes owned/being bought:",
    int(
        df["HOQ065"]
        .eq(1)
        .sum()
    )
)


print(
    "Homes rented:",
    int(
        df["HOQ065"]
        .eq(2)
        .sum()
    )
)


print(
    "Other arrangement:",
    int(
        df["HOQ065"]
        .eq(3)
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 HOQ_H XPT -> CSV
Rows: 10,175
Columns: 3
PASS: Structure = 10,175 rows x 3 columns.

--- FIELD TYPE PLAN ---
Genuine decimal variables: []
Whole-number/code variables: ['SEQN', 'HOD050', 'HOQ065']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: HOQ_H contains no tiny artifacts and no ordinary zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
HOD050: 0 fractional observations
HOQ065: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 73557
Maximum: 83731
Missing: 0
Duplicates: 0
PASS: SEQN 73557-83731 validated.

PASS: All 3 HOQ_H variables converted to nullable Int64 without rounding.

--- HOD050 ---
Observed: {np.int64(1): 88, np.int64(2): 202, np.int64(3): 683, np.int64(4): 1613, np.int64(5): 2093, np.int64(6): 1922, np.int64(7): 1272, np.int64(8): 853, np.int64(9): 574, np.int64(10): 343, np.int64(11): 197, np.int64(12): 108, np.int64(13): 85, np.int64(777): 16, np.int